# M7 · ⑦ 용어사전으로 검색어 확장
M6에서 고정한 질문과 정답을 그대로 쓴다. ⑦ 유의어 CSV에서 질문에 나온 표현을 찾고 대표 용어를 덧붙인다. 원문 질문을 지우지 않는다.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import chromadb
from sklearn.feature_extraction.text import HashingVectorizer
from IPython.display import display
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
P=ROOT/'data'/'personal'; E=P/'evaluation'; E.mkdir(parents=True,exist_ok=True)
meta=json.loads((P/'corpus'/'index_meta.json').read_text(encoding='utf-8'))
collection=chromadb.PersistentClient(path=str(ROOT/'chroma_db'/'personal')).get_collection(meta['collection'],embedding_function=None)
vectorizer=HashingVectorizer(analyzer='char',ngram_range=(2,4),n_features=1024,alternate_sign=False,norm='l2',dtype=np.float32)
def search(query,k=5,source_type='kosha_guide'):
    vector=vectorizer.transform([query]).toarray().astype(np.float32)
    found=collection.query(query_embeddings=vector.tolist(),n_results=min(100,collection.count()),
                           where={'source_type':source_type},include=['metadatas','distances'])
    hits=[];seen=set()
    for doc,distance in zip(found['metadatas'][0],found['distances'][0]):
        if doc['doc_id'] in seen:continue
        hits.append({'doc_id':doc['doc_id'],'score':round(1-float(distance),4)})
        seen.add(doc['doc_id'])
        if len(hits)==k:break
    return hits
def measure(rows):
    details=[]
    for row in rows:
        hits=search(row['query'],source_type=row['source_type'])
        ranked=[h['doc_id'] for h in hits]
        gold=set(row['gold_doc_ids'].split('|'))
        found=[i+1 for i,doc_id in enumerate(ranked) if doc_id in gold]
        details.append({'qid':row['qid'],'query':row['query'],'gold':row['gold_doc_ids'],
                        'top5':'|'.join(ranked),'hit5':int(bool(found)),
                        'precision5':len(found)/5,'rr':1/min(found) if found else 0})
    result=pd.DataFrame(details)
    scores={name:round(float(result[name].mean()),4) for name in ('hit5','precision5','rr')}
    return result,scores

## ⑦ 사전에서 쓸 표현 확인

In [ ]:
terms_path=next((P/'raw'/'source_07').glob('*유의어*.csv'))
terms=pd.read_csv(terms_path,encoding='utf-8-sig')
terms=terms.dropna(how='all')
print('사전 행',len(terms),'열',len(terms.columns))
pd.DataFrame([['포크리프트','지게차'],['모바일크레인','이동식크레인'],['백호','굴착기계']],
             columns=['질문 표현','사전 대표 용어']).pipe(display)

## 사전 실제 행에서만 확장

In [ ]:
def dictionary_expansions(query):
    # 이 자료에서 확인한 장비 표현만 적용한다. 일반 단어는 엉뚱한 문서로 갈 수 있다.
    pairs=[('포크리프트','지게차'),('모바일크레인','이동식크레인'),('백호','굴착기계')]
    additions=[]
    for alias,canonical in pairs:
        if alias not in query or canonical in query:continue
        same_row=any(alias in [str(v).strip() for v in row.dropna().tolist()]
                     and canonical in [str(v).strip() for v in row.dropna().tolist()]
                     for _,row in terms.iterrows())
        if same_row:additions.append(canonical)
    return additions
questions=pd.read_csv(E/'questions.csv',encoding='utf-8-sig').to_dict('records')
expanded=[]
for row in questions:
    extra=dictionary_expansions(row['query'])
    expanded.append({**row,'query':row['query']+(' '+ ' '.join(extra) if extra else ''),
                     'original_query':row['query'],'added_terms':'|'.join(extra)})
display(pd.DataFrame(expanded)[['qid','original_query','added_terms','query']])

## 같은 지표로 전후 비교

In [ ]:
baseline=json.loads((E/'baseline_scores.json').read_text(encoding='utf-8'))
assert baseline['index']==meta['collection'] and baseline['questions']==len(questions)
after,after_scores=measure(expanded)
before=pd.read_csv(E/'baseline_details.csv',encoding='utf-8-sig')
comparison=pd.DataFrame([{'metric':name,'before':baseline['scores'][name],
                          'after':after_scores[name],
                          'difference':round(after_scores[name]-baseline['scores'][name],4)}
                         for name in ('hit5','precision5','rr')])
display(comparison)
display(after[['qid','query','gold','top5','hit5','rr']])
comparison.to_csv(E/'synonym_comparison.csv',index=False,encoding='utf-8-sig')
after.to_csv(E/'synonym_details.csv',index=False,encoding='utf-8-sig')

## 평가 지표 전후 비교 그래프

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import Image, display

ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
if any(font.name=='Malgun Gothic' for font in font_manager.fontManager.ttflist):
    plt.rcParams['font.family']='Malgun Gothic'
plt.rcParams['axes.unicode_minus']=False
comparison=pd.read_csv(ROOT/'data'/'personal'/'evaluation'/'synonym_comparison.csv',encoding='utf-8-sig')
x=range(len(comparison));width=.36
fig,ax=plt.subplots(figsize=(8,5))
ax.bar([i-width/2 for i in x],comparison['before'],width,label='확장 전',color='#6a8fa4')
ax.bar([i+width/2 for i in x],comparison['after'],width,label='확장 후',color='#c2764d')
ax.set(xticks=list(x),xticklabels=['Hit@5','Precision@5','MRR'],ylim=(0,.8),
       ylabel='점수',title='⑧ 지침 10문항 · ⑦ 유의어 확장 전후')
ax.legend()
for container in ax.containers:ax.bar_label(container,fmt='%.3f',padding=3)
fig.tight_layout()
path=ROOT/'data'/'personal'/'figures'/'rag_synonym_before_after.png'
path.parent.mkdir(parents=True,exist_ok=True)
fig.savefig(path,dpi=170,bbox_inches='tight');plt.close(fig)
display(Image(filename=str(path)))

## 개선 실험 결과

- ⑦ 용어사전에서 확인한 세 쌍(포크리프트→지게차, 모바일크레인→이동식크레인, 백호→굴착기계)만 확장에 사용했다. 질문·정답·자료 필터·평가 방식은 기준선과 동일하게 유지했다.
- 10개 안전지침 질문에서 Hit@5는 0.60에서 0.70으로, Precision@5는 0.12에서 0.14로, 평균 reciprocal rank는 0.395에서 0.495로 바뀌었다. 이 세 지표 모두 개선됐지만 표본은 10문항뿐이다.
- 유의어를 무조건 확장하면 검색 후보가 늘어 무관 문서가 상위에 들어올 수 있다. 이번 결과만으로 모든 용어·질문에서 유의어 확장이 효과 있다고 일반화하지 않는다.

### 다음에 다시 실험할 때

- 평가 질문과 정답 문서는 고정하고 확장 규칙만 바꿔 비교한다.
- 향상된 질문과 악화된 질문을 함께 확인하고, 확장 사전의 출처와 적용된 쿼리를 기록한다.
- BM25·pgvector의 다자료 비교 결과와 M7의 10문항 가이드 실험은 평가 범위가 다르므로 수치를 한 표본처럼 합치지 않는다.
